# 04 - Qualitative coding

Combines the LLM judge's `type` labels with the human annotators' codings
to produce the qualitative codebook in the paper:

* % of capitulating responses that **paraphrase** the attacker's reasoning
* % that **invent independent** justifications
* % that **simply concede** without reasoning

Reports both judge labels (large N) and human labels (50, with kappa).

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..').resolve()))
import pandas as pd
from src.utils.io import load_jsonl

In [2]:
RUNS_DIR = pathlib.Path('../../runs')
judge = []
for jf in RUNS_DIR.rglob('judge.jsonl'):
    judge.extend(load_jsonl(jf))
j = pd.DataFrame(judge)
j = j[j.get('valid', True) == True]
j['judge_type'].value_counts(normalize=True)

judge_type
concede        0.460630
n/a            0.215551
paraphrase     0.172244
independent    0.151575
Name: proportion, dtype: float64

In [3]:
# Type breakdown by attack condition (judge labels)
breakdown = (j[j['any_turn_capitulated']]
             .groupby(['condition', 'judge_type']).size().unstack(fill_value=0))
breakdown_pct = breakdown.div(breakdown.sum(axis=1), axis=0) * 100
breakdown_pct.round(1)

judge_type,concede,independent,paraphrase
condition,,,
bare,67.5,24.2,8.3
cot,51.5,15.3,33.2


In [4]:
human_csvs = list(RUNS_DIR.rglob('annotations_*_v2.csv'))
if not human_csvs:
    human_csvs = list(RUNS_DIR.rglob('annotations_*.csv'))
if human_csvs:
    hs = pd.concat([pd.read_csv(c) for c in human_csvs], ignore_index=True)
    cap_col = 'any_turn_capitulated' if 'any_turn_capitulated' in hs.columns else 'capitulated'
    hs = hs[hs[cap_col].astype(str) == '1']
    print('Human-coded breakdown by attack condition (any-turn capitulations):')
    if len(hs):
        print((hs.groupby(['condition', 'type']).size().unstack(fill_value=0)
                 .pipe(lambda d: d.div(d.sum(axis=1), axis=0) * 100).round(1)))
    else:
        print('  (no capitulations in human annotation set)')
else:
    print('No human annotations yet -- generate them via scripts/make_human_eval.py')


Human-coded breakdown by attack condition (any-turn capitulations):
type       concede  independent  paraphrase
condition                                  
bare          64.7         29.4         5.9
cot           59.1          0.0        40.9
